# DiCo-NLI - mmBERT Merged-Track Training

**Repository:** https://github.com/AmnO-O/DICO-NLI

Fine-tunes **`jhu-clsp/mmBERT-base`** (a multilingual ModernBERT covering EN/ES/EU and 1800+
languages) on the DiCo-NLI data and compares two training strategies on the same seeds:

| Mode | Training data | Evaluation |
|------|---------------|------------|
| `per_track` | one track's train split | that track's dev split |
| `merged` | all four tracks concatenated | each track's dev split, separately |

- backbones: `jhu-clsp/mmBERT-base`, `jhu-clsp/mmBERT-small`
- seeds: `2021, 2022, 2023, 2024, 2025`
- tracks: `1 (EN), 2 (ES), 3 (EU), 4 (mixed)`

All four tracks are built from the **same 1,760 source pairs**, rendered in different
language combinations, and share an identical label distribution. Merging therefore tests
**cross-lingual transfer**, not extra supervision. See the project README for the task.

## Why mmBERT

`mmBERT-base` is a `modernbert` architecture (`model_type: modernbert`), so it loads natively
with the starter kit's pinned `transformers==4.48.3` via `AutoModelForSequenceClassification`.
Unlike the English-only ModernBERT, it covers Spanish and Basque.

| Model | Total params | Non-embed | Hidden | Heads | Layers |
|---|---:|---:|---:|---:|---:|
| mmBERT-small | 140M | 42M | 384 | 6 | 22 |
| mmBERT-base | 307M | 110M | 768 | 12 | 22 |

## How a run works

1. Train via `starter_kit/main.py` with `--save_path` (no dev during training).
2. Load the saved checkpoint and run inference on the relevant dev split(s).
3. Score with the official `evaluation_functions` scorer.

Each training run also writes a **per-epoch history** (`<run_dir>/training_history.json`,
with a `.csv` companion) recording the average training loss, learning rate, and dev
metrics when available - so you can inspect how the model is learning (section 5).

Both modes share the **exact same inference + scoring path**, so `per_track` vs `merged`
comparisons are apples-to-apples.

Runs are **resumable** (a saved checkpoint and scored rows are reused). Set
`OVERWRITE = True` to force a clean re-run.

## Data location (auto-detected)

- **Kaggle:** `/kaggle/input/datasets/ieltsmater/dico-nli/DiCO-NLI` (must contain `train/`, `dev/`)
- **Local:** `<repo>/final_data`

If the code repo is not present (fresh Kaggle notebook), the setup cell `git clone`s it into
`/kaggle/working/DiCO-NLI`; enable **Internet** in the notebook settings for that to work.

## Prerequisites

```bash
pip install -r starter_kit/requirements.txt
```

GPU strongly recommended. `mmBERT-base` is ~307M params; on a 16GB GPU keep
`BATCH_SIZE <= 16`. Trim `SEEDS`/`TRACKS`/`ACTIVE_MODELS` for a quick smoke test.

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain-python fallback (nbconvert / script)
    def display(obj):
        print(obj)


GITHUB_REPO = "https://github.com/AmnO-O/DICO-NLI"
GITHUB_BRANCH = "main"

# Kaggle paths (only active when running on Kaggle's Linux environment).
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/ieltsmater/dico-nli/DiCO-NLI")
IS_KAGGLE = sys.platform.startswith("linux") and KAGGLE_WORKING.is_dir()

# Set to a path to force a specific data directory, or leave None to auto-detect.
DATA_ROOT_OVERRIDE = None

# Class-index order used by the starter kit's sklearn LabelEncoder
# (starter_kit/src/trainer/helpers/data_reader.py). Required to map model logits
# back to the official label strings during custom inference.
CLASS_ORDER = ["BACKWARD_ENTAILMENT", "EQUIVALENCE", "FORWARD_ENTAILMENT", "NEGATIVE_OTHER"]


def is_repo_root(path: Path) -> bool:
    return (path / "starter_kit" / "main.py").exists() and (path / "evaluation_functions").is_dir()


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if is_repo_root(candidate):
            return candidate
    return None


def ensure_repo() -> Path:
    # 1) a checkout is already available (local machine, or bundled in the Kaggle dataset)
    candidates = [Path.cwd().resolve()]
    if IS_KAGGLE:
        candidates += [KAGGLE_DATA_ROOT, KAGGLE_WORKING / "DiCO-NLI"]
    for candidate in candidates:
        if candidate.exists():
            root = find_repo_root(candidate)
            if root is not None:
                return root
    # 2) otherwise clone it (Kaggle: enable Internet in the notebook settings)
    clone_dir = (KAGGLE_WORKING if IS_KAGGLE else Path.cwd()) / "DiCO-NLI"
    if not is_repo_root(clone_dir):
        print(f"Cloning {GITHUB_REPO} -> {clone_dir}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(clone_dir)],
            check=True,
        )
    return clone_dir


def resolve_data_root() -> Path:
    if DATA_ROOT_OVERRIDE is not None:
        return Path(DATA_ROOT_OVERRIDE)
    if IS_KAGGLE and (KAGGLE_DATA_ROOT / "train").is_dir() and (KAGGLE_DATA_ROOT / "dev").is_dir():
        return KAGGLE_DATA_ROOT
    return REPO_ROOT / "final_data"


REPO_ROOT = ensure_repo()
STARTER_KIT = REPO_ROOT / "starter_kit"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_functions import score_files  # noqa: E402

DATA_ROOT = resolve_data_root()

print("Python  :", sys.executable)
print("Kaggle  :", IS_KAGGLE)
print("Repo    :", REPO_ROOT)
print("GitHub  :", GITHUB_REPO)
print("Data    :", DATA_ROOT)
print("Train   :", (DATA_ROOT / "train").exists())
print("Dev     :", (DATA_ROOT / "dev").exists())


def run_logged_command(cmd, cwd, log_path, echo=("Epoch", "average loss", "Training history")):
    # Run a command, tee its stdout/stderr into log_path, and echo the lines that
    # matter (per-epoch loss) live so you can watch training progress.
    with Path(log_path).open("w", encoding="utf-8") as log:
        log.write(" ".join(cmd) + "\n\n")
        log.flush()
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if any(token in line for token in echo):
                print("    " + line.rstrip())
        returncode = proc.wait()
    return returncode

In [ ]:
# ----------------------------- experiment grid -----------------------------
MODELS = {
    "mmBERT-small": "jhu-clsp/mmBERT-small",
    "mmBERT-base": "jhu-clsp/mmBERT-base",
}
ACTIVE_MODELS = ["mmBERT-base"]        # add "mmBERT-small" to compare sizes
MODES = ["per_track", "merged"]        # both are always compared
SEEDS = [2021, 2022, 2023, 2024, 2025]
TRACKS = [1, 2, 3, 4]

# ----------------------------- hyperparameters -----------------------------
# These must satisfy starter_kit/main.py validation rules.
MAX_LENGTH = 256
NUM_EPOCHS = 4
FREEZE_BOTTOM_LAYERS = None  # e.g., 18 to freeze bottom 18/22, train only top 4 + classifier
LR = 5e-5          # allowed: 1e-6, 5e-6, 8e-6, 9e-6, 1e-5, 1.5e-5, 2e-5, 2.5e-5, 3e-5, 4e-5, 5e-5, 8e-5, 1e-4
BATCH_SIZE = 16    # multiple of 8, 8..64 (mmBERT-base is 307M -> keep <= 16 on 16GB GPUs)
WD = 1e-5          # allowed: 1e-6, 5e-6, 8e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1
WARMUP_PCT = 0.01  # 0 .. 0.1, step 0.01
DROPOUT = 0.1      # 0 .. 0.5

# ----------------------------- output locations ----------------------------
OUTPUT_ROOT = KAGGLE_WORKING if IS_KAGGLE else REPO_ROOT
RESULTS_DIR = OUTPUT_ROOT / "results" / "mmbert_merged"
LOG_DIR = RESULTS_DIR / "logs"
SUMMARY_CSV = RESULTS_DIR / "scores.csv"
MERGED_DIR = RESULTS_DIR / "merged"
MERGED_TRAIN = MERGED_DIR / "dico_nli_train_all_participant_labeled.csv"
OVERWRITE = False  # True -> retrain and rebuild everything

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

PLANNED_TRAIN_RUNS = len(ACTIVE_MODELS) * (len(TRACKS) + 1) * len(SEEDS)
print(f"{len(ACTIVE_MODELS)} model(s) x (per_track x {len(TRACKS)} + merged x 1) x {len(SEEDS)} seeds "
      f"= {PLANNED_TRAIN_RUNS} training runs")
print("Results ->", RESULTS_DIR)
print("Merged train cache ->", MERGED_TRAIN)

## 2. Helper functions

In [ ]:
def track_paths(track: int):
    train = DATA_ROOT / "train" / f"dico_nli_train_track{track}_participant_labeled.csv"
    dev = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_participant_labeled.csv"
    ref = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_reference.csv"
    for path in (train, dev, ref):
        if not path.exists():
            raise FileNotFoundError(path)
    return train, dev, ref


def ensure_merged_train() -> Path:
    if MERGED_TRAIN.exists() and not OVERWRITE:
        return MERGED_TRAIN
    MERGED_TRAIN.parent.mkdir(parents=True, exist_ok=True)
    frames = [pd.read_csv(track_paths(t)[0]) for t in TRACKS]
    merged = pd.concat(frames, ignore_index=True)
    merged.to_csv(MERGED_TRAIN, index=False)
    counts = merged["label"].value_counts().to_dict()
    print(f"Merged train: {len(merged)} rows from {len(TRACKS)} tracks -> {MERGED_TRAIN}")
    print("Label distribution:", counts)
    return MERGED_TRAIN


def slug(name: str) -> str:
    return name.lower().replace(".", "")


def build_train_command(model_id, train_path, seed, run_dir, model_dir):
    return [
        sys.executable, "-u", "main.py",
        "--model_path", model_id,
        "--tokenizer_path", model_id,
        "--do_train",
        "--train_path", str(train_path),
        "--no-do_dev",
        "--no-do_test",
        "--seed", str(seed),
        "--dropout", str(DROPOUT),
        "--warmup_pcrt", str(WARMUP_PCT),
        "--lr", str(LR),
        "--batch_size", str(BATCH_SIZE),
        "--wd", str(WD),
        "--num_epochs", str(NUM_EPOCHS),
        "--no-is_optuna_trial",
        "--max_length", str(MAX_LENGTH),
"--results_dir", str(run_dir),
"--save_path", str(model_dir),
*(["--freeze_bottom_layers", str(FREEZE_BOTTOM_LAYERS)] if FREEZE_BOTTOM_LAYERS is not None else []),
    ]


def train_model(model_id, train_path, seed, run_dir: Path, log_path: Path) -> bool:
    model_dir = run_dir / "model"
    if model_dir.exists() and not OVERWRITE:
        print(f"[skip] {run_dir.name} (checkpoint on disk)")
        return True
    if OVERWRITE and run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    cmd = build_train_command(model_id, train_path, seed, run_dir, model_dir)
    print(f"[run ] {run_dir.name}")
    started = time.time()
    returncode = run_logged_command(cmd, STARTER_KIT, log_path)
    elapsed = time.time() - started

    if returncode != 0 or not model_dir.exists():
        print(f"[fail] {run_dir.name}: exit {returncode}; see {log_path}")
        return False
    print(f"[ok  ] {run_dir.name} trained in {elapsed / 60:.1f} min")
    return True


def score_predictions(key, model_key, model_id, mode, track, seed, pred_path, ref_path, run_dir, scores_subdir):
    scores_dir = run_dir / scores_subdir
    report = score_files(ref_path, pred_path, output_dir=scores_dir)
    return {
        "run": key,
        "model": model_key,
        "model_id": model_id,
        "mode": mode,
        "track": track,
        "seed": seed,
        "weighted_f1": report.weighted_f1,
        "soft_cons": report.soft_cons,
        "hard_cons": report.hard_cons,
        "macro_f1": report.classification.macro_f1,
        "accuracy": report.classification.accuracy,
        "instances": report.classification.total_instances,
        "predictions": str(pred_path),
        "scores_dir": str(scores_dir),
    }

In [ ]:
def predict_dev(model_dir: Path, dev_path: Path, out_path: Path) -> Path:
    # Custom inference mirroring starter_kit Model.tokenize (pairs of text1, text2).
    import re
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    dev = pd.read_csv(dev_path)

    def clean(value):
        return re.sub(r"\s+", " ", str(value)).strip()

    text1 = [clean(v) for v in dev["text1"].tolist()]
    text2 = [clean(v) for v in dev["text2"].tolist()]

    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    preds = []
    with torch.no_grad():
        for start in range(0, len(text1), BATCH_SIZE):
            batch = tokenizer(
                text1[start:start + BATCH_SIZE],
                text2[start:start + BATCH_SIZE],
                padding="max_length",
                truncation="only_second",
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            batch = {key: value.to(device) for key, value in batch.items()}
            preds.extend(model(**batch).logits.argmax(dim=-1).cpu().tolist())

    out = pd.DataFrame({"instance_id": dev["instance_id"], "label": [CLASS_ORDER[p] for p in preds]})
    out.to_csv(out_path, index=False)
    return out_path


def run_per_track(model_key, model_id, track, seed):
    train, dev, ref = track_paths(track)
    key = f"{slug(model_key)}_per_track_track{track}_seed{seed}"
    run_dir = RESULTS_DIR / key

    if not train_model(model_id, train, seed, run_dir, LOG_DIR / f"{key}_train.log"):
        return None

    pred_path = run_dir / "predictions_dev.csv"
    if not pred_path.exists() or OVERWRITE:
        predict_dev(run_dir / "model", dev, pred_path)
    return score_predictions(key, model_key, model_id, "per_track", track, seed,
                             pred_path, ref, run_dir, "official_scores")


def run_merged(model_key, model_id, seed, tracks=None):
    tracks = list(tracks if tracks is not None else TRACKS)
    merged_train = ensure_merged_train()
    key = f"{slug(model_key)}_merged_seed{seed}"
    run_dir = RESULTS_DIR / key

    if not train_model(model_id, merged_train, seed, run_dir, LOG_DIR / f"{key}_train.log"):
        return None

    rows = []
    for track in tracks:
        _, dev, ref = track_paths(track)
        pred_path = run_dir / f"predictions_dev_track{track}.csv"
        if not pred_path.exists() or OVERWRITE:
            predict_dev(run_dir / "model", dev, pred_path)
        rows.append(score_predictions(f"{key}_track{track}", model_key, model_id, "merged", track, seed,
                                      pred_path, ref, run_dir, f"official_scores_track{track}"))
    return rows

## 3. Run the experiment grid

In [ ]:
results, done = [], set()

if SUMMARY_CSV.exists() and not OVERWRITE:
    existing = pd.read_csv(SUMMARY_CSV)
    if "mode" in existing.columns:
        results = existing.to_dict("records")
        done = {(r["model"], r["mode"], int(r["track"]), int(r["seed"])) for r in results}
    else:
        print("Existing summary lacks a 'mode' column; starting fresh.")
elif OVERWRITE:
    print("OVERWRITE=True: starting fresh.")


def flush():
    pd.DataFrame(results).to_csv(SUMMARY_CSV, index=False)


print(f"{len(done)} scored rows already on disk\n")

for model_key in ACTIVE_MODELS:
    model_id = MODELS[model_key]
    for seed in SEEDS:
        # --- per-track ---
        for track in TRACKS:
            if (model_key, "per_track", track, seed) in done:
                continue
            try:
                row = run_per_track(model_key, model_id, track, seed)
            except Exception:
                traceback.print_exc()
                row = None
            if row is not None:
                results.append(row)
                done.add((model_key, "per_track", track, seed))
                flush()
                print(f"  per_track t{track} s{seed}: "
                      f"f1={row['weighted_f1']:.4f} soft={row['soft_cons']:.4f} hard={row['hard_cons']:.4f}")

        # --- merged (one training run covers every pending track) ---
        pending = [t for t in TRACKS if (model_key, "merged", t, seed) not in done]
        if pending:
            try:
                rows = run_merged(model_key, model_id, seed, tracks=pending)
            except Exception:
                traceback.print_exc()
                rows = None
            for row in rows or []:
                results.append(row)
                done.add((model_key, "merged", row["track"], seed))
                print(f"  merged    t{row['track']} s{seed}: "
                      f"f1={row['weighted_f1']:.4f} soft={row['soft_cons']:.4f} hard={row['hard_cons']:.4f}")
            if rows:
                flush()

df = pd.DataFrame(results)
print(f"\nTotal scored rows: {len(df)}")

## 4. Results: per-track vs merged

In [ ]:
df = pd.read_csv(SUMMARY_CSV)
df["track"] = df["track"].astype(int)
df["seed"] = df["seed"].astype(int)

summary = (
    df.groupby(["model", "mode", "track"], as_index=False)
    .agg(
        n=("seed", "count"),
        weighted_f1_mean=("weighted_f1", "mean"), weighted_f1_std=("weighted_f1", "std"),
        soft_cons_mean=("soft_cons", "mean"), soft_cons_std=("soft_cons", "std"),
        hard_cons_mean=("hard_cons", "mean"), hard_cons_std=("hard_cons", "std"),
    )
)
summary.to_csv(RESULTS_DIR / "summary_by_mode_track.csv", index=False)
print("Mean +/- std over seeds (per model / mode / track):")
display(summary.round(4))

macro = (
    df.groupby(["model", "mode"], as_index=False)
    .agg(
        weighted_f1=("weighted_f1", "mean"),
        soft_cons=("soft_cons", "mean"),
        hard_cons=("hard_cons", "mean"),
    )
)
macro.to_csv(RESULTS_DIR / "summary_macro.csv", index=False)
print("\nMacro-average across tracks:")
display(macro.round(4))

In [ ]:
for metric in ["weighted_f1", "soft_cons", "hard_cons"]:
    print(f"{metric}: rows = model/track, columns = mode (mean over seeds)")
    display(df.pivot_table(index=["model", "track"], columns="mode", values=metric).round(4))

In [ ]:
# Per-seed breakdown for each active model (weighted_f1).
for model_key in ACTIVE_MODELS:
    sub = df[df["model"] == model_key]
    if sub.empty:
        continue
    print(f"{model_key}: weighted_f1 by mode/track x seed")
    display(sub.pivot_table(index=["mode", "track"], columns="seed", values="weighted_f1").round(4))

## 5. Training curves (per-epoch loss)

In [ ]:
# Every training run writes a per-epoch history to <run_dir>/training_history.json
# (and a matching .csv). The starter kit's trainer records the average training loss
# per epoch, the learning rate, and - when dev evaluation is enabled - the dev metrics.
def load_history(run_dir):
    path = Path(run_dir) / "training_history.json"
    if not path.exists():
        return None
    return pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))


def find_histories():
    return sorted(RESULTS_DIR.glob("*/training_history.json"))


available = find_histories()
print(f"{len(available)} run(s) with a recorded training history:")
for path in available:
    print(" -", path.parent.name)

In [ ]:
# Inspect a single run's learning curve (defaults to the first available run).
RUN_TO_PLOT = available[0].parent.name if available else None
print("Selected:", RUN_TO_PLOT)

hist = load_history(RESULTS_DIR / RUN_TO_PLOT) if RUN_TO_PLOT else None
if hist is None:
    print("No training history yet - run the training grid first.")
else:
    display(hist)
    try:
        import matplotlib.pyplot as plt

        fig, ax1 = plt.subplots(figsize=(7, 4))
        ax1.plot(hist["epoch"], hist["train_loss"], marker="o", color="tab:red")
        ax1.set_xlabel("epoch")
        ax1.set_ylabel("train loss", color="tab:red")
        ax1.tick_params(axis="y", labelcolor="tab:red")
        if "dev_f1_weighted" in hist.columns:
            ax2 = ax1.twinx()
            ax2.plot(hist["epoch"], hist["dev_f1_weighted"], marker="s", color="tab:blue")
            ax2.set_ylabel("dev weighted F1", color="tab:blue")
            ax2.tick_params(axis="y", labelcolor="tab:blue")
        ax1.set_title(f"{RUN_TO_PLOT}: loss curve")
        fig.tight_layout()
        plt.show()
    except ImportError:
        print("(matplotlib not installed - the table above has the per-epoch values)")

In [ ]:
# Overlay every run's train loss, and show the raw log tail of a chosen run.
if available:
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(9, 5))
        for path in available:
            h = load_history(path.parent)
            ax.plot(h["epoch"], h["train_loss"], marker="o", markersize=3, label=path.parent.name)
        ax.set_xlabel("epoch")
        ax.set_ylabel("train loss")
        ax.set_title("Train loss per run")
        ax.legend(fontsize=7)
        plt.show()
    except ImportError:
        print("(matplotlib not installed)")

RUN_TO_INSPECT = available[0].parent.name if available else None
log_path = LOG_DIR / f"{RUN_TO_INSPECT}_train.log" if RUN_TO_INSPECT else None
if log_path is not None and log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"\n{log_path.name} ({len(lines)} lines) - last 40:")
    print("\n".join(lines[-40:]))
else:
    print("No training log yet:", log_path)

## 6. Inspect the best run

In [ ]:
best = df.sort_values(["hard_cons", "weighted_f1"], ascending=False).iloc[0]
scores = json.loads((Path(best["scores_dir"]) / "scores.json").read_text(encoding="utf-8"))

print("Run   :", best["run"])
print("Mode  :", best["mode"], "| Track:", best["track"], "| Seed:", best["seed"])
print("Weighted F1:", round(scores["weighted_f1"], 4))
print("SoftCons   :", round(scores["soft_cons"], 4))
print("HardCons   :", round(scores["hard_cons"], 4))

print("\nPer-label metrics:")
display(pd.DataFrame(scores["classification"]["per_label"]).T)

print("\nConfusion matrix (rows = gold, cols = predicted):")
display(pd.DataFrame(scores["classification"]["confusion_matrix"]))

print("\nSample pair errors (first 10):")
display(pd.DataFrame(scores["consistency"]["pair_errors"]).head(10))

In [ ]:
# (Optional) run a single configuration directly.

# Per-track example:
row = run_per_track("mmBERT-base", MODELS["mmBERT-base"], track=3, seed=2021)
display(pd.Series(row).to_frame().T) if row else print("run failed")

# Merged example (trains once, evaluates every track):
# rows = run_merged("mmBERT-base", MODELS["mmBERT-base"], seed=2021)
# display(pd.DataFrame(rows))